In [1]:
import sys
import warnings
import numpy as np
import pathlib as pl
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm
from scipy.optimize import minimize
root_folder = pl.Path.cwd().parents[2]
sys.path.insert(0, str(root_folder / "utilities"))
import common_functions as cf

warnings.filterwarnings('ignore')

initial_data_folder = "data/initial_data/function_5"
initial_inputs_path = pl.Path.joinpath(root_folder, initial_data_folder,  "initial_inputs.npy")
initial_outputs_path = pl.Path.joinpath(root_folder, initial_data_folder, "initial_outputs.npy")

In [2]:
data_in = np.load(initial_inputs_path)
data_out = np.load(initial_outputs_path)

Week-01

In [3]:
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(data_in, data_out)

# Surrogate to maximise (negative for minimize)
def surrogate_neg(x):
    return -gp.predict(x.reshape(1, -1))[0]

# Bounds for normalized inputs
bounds = [(0,1), (0,1), (0,1), (0,1)]

# Try multiple random starts to avoid local issues
best_x = None
best_val = float('inf')
for _ in range(10):
    x0 = np.random.rand(4)
    res = minimize(surrogate_neg, x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_val:
        best_val = res.fun
        best_x = res.x

x_next = best_x
print("Next point to evaluate:", x_next)

Next point to evaluate: [0.27645867 0.68188949 0.88871761 0.91060843]


Week-02

In [4]:
new_points = np.array([
    [0.232877,0.841416,0.883342,0.879464]
])

new_outputs = np.array([
    1091.3271430129832
])


# Combine all data
X_all = np.vstack([data_in, new_points])
y_all = np.concatenate([data_out, new_outputs])

# --- Refit Gaussian Process ---
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_all, y_all)

# --- Generate candidate next points around current best ---
best_x = new_points
sigma = 0.02  # tweak size for local exploration
num_candidates = 5

x_next_candidates = best_x + np.random.normal(0, sigma, size=(num_candidates, 4))
# Ensure all points are within [0,1]
x_next_candidates = np.clip(x_next_candidates, 0, 1)

print("Candidate next points to evaluate:")
print(x_next_candidates)

Candidate next points to evaluate:
[[0.26015638 0.8365662  0.87757207 0.86028091]
 [0.26267754 0.86069088 0.86718492 0.87993338]
 [0.22716543 0.85606101 0.86910081 0.87808135]
 [0.25867175 0.81286343 0.88148    0.89203022]
 [0.22534113 0.83392987 0.88643261 0.88476612]]


Week-03

In [5]:
# --- Add the two new data points ---
new_points = np.array([
    [0.232877,0.841416,0.883342,0.879464],
    [0.245154,0.843081,0.898729,0.883391]
])

new_outputs = np.array([
    1091.3271430129832,
    1195.7279770056589
])


# Combine all data
X_all = np.vstack([data_in, new_points])
y_all = np.concatenate([data_out, new_outputs])

# --- Fit updated Gaussian Process ---
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-8,                # smaller noise term for precision
    normalize_y=True,
    n_restarts_optimizer=5     # more robust kernel fitting
)
gp.fit(X_all, y_all)

# --- Define acquisition function (UCB variant) ---
def surrogate_neg_ucb(x, kappa=2.0):
    mean, std = gp.predict(x.reshape(1, -1), return_std=True)
    return -(mean + kappa * std)

# --- Search bounds for normalized inputs ---
bounds = [(0, 1), (0, 1), (0, 1), (0, 1)]

# --- Optimize acquisition function for next sampling point ---
best_x, best_val = None, float('inf')
for _ in range(10):
    x0 = np.random.rand(4)
    res = minimize(surrogate_neg_ucb, x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_val:
        best_val = res.fun
        best_x = res.x

x_next = best_x

print("Suggested next point to evaluate:", x_next)

# --- Optionally: generate a few local perturbations for fine exploration ---
sigma = 0.01
num_candidates = 5
x_next_candidates = x_next + np.random.normal(0, sigma, size=(num_candidates, 4))
x_next_candidates = np.clip(x_next_candidates, 0, 1)

print("\nLocal candidate points for fine-tuning:")
print(x_next_candidates)

res_formatted = [f"{r:.6f}" for r in x_next]
result = "-".join(res_formatted)
print(result)

x_next_6dp = np.round(x_next, 6)
x_next_6dp

Suggested next point to evaluate: [0.23728591 0.89782783 0.94744489 0.89713396]

Local candidate points for fine-tuning:
[[0.22535077 0.90818206 0.95398718 0.89799248]
 [0.22567713 0.90524799 0.95281193 0.91862741]
 [0.24079929 0.87966205 0.95423723 0.89637372]
 [0.2522895  0.9056903  0.93617311 0.89350163]
 [0.23958616 0.91191791 0.96023049 0.89667861]]
0.237286-0.897828-0.947445-0.897134


array([0.237286, 0.897828, 0.947445, 0.897134])